In [ ]:
# ! python -m pip install --no-index --find-links=/kaggle/usr/lib/pip-install-permanent/my_packages -r /kaggle/usr/lib/pip-install-permanent/requirements.txt

In [ ]:
import os
os.environ['PACKAGE_DIR'] = '/kaggle/usr/lib/pip-install-permanent'

In [ ]:
# # import helper_functions as hf
from helper_functions import *
# import nltk
# from nltk.corpus import stopwords
# from nltk.stem import WordNetLemmatizer
# import string
# from spellchecker import SpellChecker
# from textblob import TextBlob
# from multiprocessing import Pool
# from tqdm import tqdm
# import numpy as np
# import pandas as pd
# # Preprocessing
# from nltk.tokenize import word_tokenize, sent_tokenize
# import operator
# from spellchecker import SpellChecker
# from tqdm import tqdm  # Import tqdm
# import re
# import inflect
# from wordsegment import load, segment
# from nltk.corpus import words
# word_list = set(words.words())
# from spellchecker import SpellChecker

# from tqdm.contrib.concurrent import process_map  # If this import fails, you might need to update tqdm

# import multiprocessing
 
# # Import Packages
# import shutup; shutup.please()
# import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt
# import tensorflow as tf
# import keras_tuner as kt
# import seaborn as sns

# from nltk.corpus import stopwords, wordnet
# from nltk.tokenize import word_tokenize, sent_tokenize
# from nltk import pos_tag, ne_chunk
# from textblob import TextBlob

# from textstat import flesch_reading_ease, smog_index

# import spacy
# from collections import Counter
# from gensim import corpora, models
# import pyLDAvis.gensim as gen
# import pyLDAvis
# import re

# # Machine Learning & Data Preprocessing

# from sklearn.preprocessing import StandardScaler, MinMaxScaler
# from sklearn.model_selection import train_test_split
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.metrics.pairwise import cosine_similarity

# # Deep Learning

# from tensorflow.keras import layers
# from tensorflow.keras.preprocessing.text import Tokenizer
# from tensorflow.keras.preprocessing.sequence import pad_sequences

# # Gensim
# # from gensim.models import Word2Vec, KeyedVectors

# # Progress bar
# from tqdm import tqdm

# # Keras Tuner
# from keras_tuner.tuners import RandomSearch

# # # Setting logging levels and environment variables
# # tf.get_logger().setLevel(logging.ERROR)
# # os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# from textstat import flesch_reading_ease

import sklearn
print(sklearn.__version__)

In [ ]:
import pandas as pd

test = pd.read_csv('/kaggle/input/learning-agency-lab-automated-essay-scoring-2/test.csv')

test.head()

In [ ]:
import os

glove_path = '/kaggle/input/embeddings/glove-840B-300d.txt'

paragram_path = '/kaggle/input/embeddings/paragram-300-sl999.txt'

wiki_news_path = '/kaggle/input/embeddings/wiki-news-1M-300d.vec'


# Load embeddings

embeddings = parallel_load_embeddings([glove_path, paragram_path, wiki_news_path])

glove = embeddings["glove"]
paragram = embeddings["paragram"]
fasttext = embeddings["fasttext"]

In [ ]:
test, glove, paragram, fastetxt = spellcheck_and_correct_text(test, embeddings)

In [ ]:
test.head()

In [ ]:
drop_cols = [ 'full_text', 'lowered','corrected_text', 'segmented_text','dense_dense_vector']

test_df = test.copy()

test_df.drop(columns=drop_cols, inplace=True)

In [ ]:
import pandas as pd
from tqdm import tqdm
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    Trainer, 
    TrainingArguments, 
    DataCollatorWithPadding
)
from datasets import Dataset
from glob import glob
import gc
import torch
from scipy.special import softmax

MAX_LENGTH = 1024
# TEST_DATA_PATH = "/kaggle/input/learning-agency-lab-automated-essay-scoring-2/test.csv"
MODEL_PATH = '/kaggle/input/aes-deberta-large/*/*'

EVAL_BATCH_SIZE = 10  # Lower if having memory issues

# Check if GPU is available and set device accordingly
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Fetch model paths
models = glob(MODEL_PATH)

# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained(models[0])

# Tokenize and move dataset to GPU
ds = Dataset.from_pandas(test_df).map(tokenize).remove_columns(['essay_id', 'clean_text']).to(device)

# Set training arguments
args = TrainingArguments(
    ".", 
    per_device_eval_batch_size=EVAL_BATCH_SIZE, 
    report_to="none"
)

# Initialize list for predictions
predictions = []

# Loop through each model and get predictions, now with device set
for model_path in tqdm(models, desc="Processing Models"):
    # Load the model and move it to the set device (GPU/CPU)
    model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)
    
    # Create a trainer with the set device
    trainer = Trainer(
        model=model, 
        args=args, 
        data_collator=DataCollatorWithPadding(tokenizer), 
        tokenizer=tokenizer,
        device=device  # This ensures operations happen on the appropriate device
    )
    
    # Perform predictions
    preds = trainer.predict(ds).predictions
    predictions.append(softmax(preds, axis=-1))  # Apply softmax to get probabilities
    
    # Cleanup
    del model, trainer
    torch.cuda.empty_cache()
    gc.collect()

# Calculate the average prediction score
predicted_score = 0.0

# Wrap this loop with tqdm to track progress
for p in tqdm(predictions, desc="Averaging Predictions"):
    predicted_score += p
    
# Get the final average score
predicted_score /= len(predictions)

In [ ]:
print("Predicted score: \n ", predicted_score)


In [ ]:
# Add the probabilities as new columns to the training dataframe.
for i in range(predicted_score.shape[1]):
    test[f'deberta_prob_{i}'] = predicted_score[:, i]

# Display the updated dataframe
test.head()

In [ ]:
# test.columns = test.columns.str.replace("combined", "dense")

In [ ]:
scaler_path = '/kaggle/input/minmaxscaler-sklearn-model/sklearn_scaler.pkl'

In [ ]:
import pickle

# Load the scaler object from the pickle file
with open(scaler_path, 'rb') as file:
    scaler = pickle.load(file)
    

feature_cols = []

for col in test_df.columns:
    if (col != 'essay_id') and (col != 'score'):
        feature_cols.append(col) 

# Select relevant columns (replace with actual column names)

test_features = test_df[feature_cols]

# Standardize the features if needed

test_features_scaled = scaler.transform(test_features)

test_df[feature_cols] = test_features_scaled  # Replace the original features with scaled ones


In [ ]:
# Check for the existence of the file
if not os.path.exists(scaler_path):
    print(f"File {scaler_path} does not exist.")
elif os.path.getsize(scaler_path) == 0:
    # If the file exists but its size is 0, it's empty
    print(f"File {scaler_path} exists but is empty.")
else:
    # If the file exists and is not empty
    print(f"File {scaler_path} exists and is not empty.")
    
    
# Check if the scaler is StandardScaler
if isinstance(scaler, StandardScaler):
    print("The scaler is a StandardScaler.")
elif isinstance(scaler, MinMaxScaler):
    print("The scaler is a MinMaxScaler.")
else:
    print("The scaler is neither StandardScaler nor MinMaxScaler.")

In [ ]:
test_df = pca_dataframe(test_df)


In [ ]:
test_features = test_df.drop(columns=['essay_id', 'cosine_max'], axis=1).values



print("Features shape:", test_features.shape)


In [ ]:
def quadratic_weighted_kappa_scorer(y_true, y_pred):
    """
    Compute the Quadratic Weighted Kappa (QWK), also known as Cohen's kappa.
    
    Parameters:
    y_true : array-like of shape (n_samples,)
        True labels.
    y_pred : array-liimport keras_tuner
from sklearn import ensemble
from sklearn import datasets
from sklearn import linear_model
from sklearn import metrics
from sklearn import model_selection

def build_model(hp):
  model_type = hp.Choice('model_type', ['random_forest', 'ridge'])
  if model_type == 'random_forest':
    model = ensemble.RandomForestClassifier(
        n_estimators=hp.Int('n_estimators', 10, 50, step=10),
        max_depth=hp.Int('max_depth', 3, 10))
  else:
    model = linear_model.RidgeClassifier(
        alpha=hp.Float('alpha', 1e-3, 1, sampling='log'))
  return model

tuner = keras_tuner.tuners.SklearnTuner(
    oracle=keras_tuner.oracles.BayesianOptimizationOracle(
        objective=keras_tuner.Objective('score', 'max'),
        max_trials=100),
    hypermodel=build_model,
    scoring=metrics.make_scorer(metrics.accuracy_score),
    cv=model_selection.StratifiedKFold(10),
    directory='.',
    project_name='my_project')ke of shape (n_samples,)
        Predicted labels.
    
    Returns:
    score : float
        Quadratic Weighted Kappa score.
    """
    return cohen_kappa_score(y_true, y_pred, weights='quadratic')

In [ ]:
# import shutil

# # Path to the file in the read-only input directory
# input_path = '/kaggle/usr/lib/sklearn_model/working/standard_random_forest.joblib'

# # Path where the file should be copied to in the writable working directory
# output_path = '/kaggle/working/standard_random_forest.pkl'

# # Copying the file
# shutil.copy(input_path, output_path)

# # Now the file is available in the working directory and can be modified or used as needed.


In [ ]:
# # Navigate to the directory with your .whl files
# !cd /kaggle/input/sklearn-1-4-1/sklearn_package

# # Install the packages with pip
# !pip install /kaggle/input/sklearn-1-4-1/sklearn_package/joblib-1.4.0-py3-none-any.whl
# !pip install /kaggle/input/sklearn-1-4-1/sklearn_package/numpy-1.26.4-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
# !pip install /kaggle/input/sklearn-1-4-1/sklearn_package/scipy-1.13.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
# !pip install /kaggle/input/sklearn-1-4-1/sklearn_package/threadpoolctl-3.4.0-py3-none-any.whl
# !pip install /kaggle/input/sklearn-1-4-1/sklearn_package/scikit_learn-1.4.1.post1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl


In [ ]:
from joblib import dump, load
import sklearn_model

forest_model = load('/kaggle/usr/lib/sklearn_model/working/standard_random_forest.joblib')



In [ ]:
# # Path to the file in the read-only input directory
# model_input_path = ' '

# # Path where the file should be copied to in the writable working directory
# model_output_path = '/kaggle/working/best_standard_model.keras'

# # Copying the file
# shutil.copy(model_input_path, model_output_path)

In [ ]:
# Load the previously saved model

# model_path ='/kaggle/working/best_standard_model.keras'

# loaded_model = tf.keras.models.load_model(model_path)

# Now, `loaded_model` is ready to be used for inference

test_predictions = forest_model.predict(test_features)

# `test_predictions` will contain the output from the model
# Depending on your model's output, you might want to process these predictions further,
# like converting them from probabilities to class labels if it's a classification model

# predicted_classes = np.argmax(test_predictions, axis=1)

In [ ]:
test_predictions

In [ ]:
# predicted_classes = predicted_classes + 1
# predicted_classes

In [ ]:
submission = test[['essay_id']].copy()

submission['score'] = test_predictions


submission

In [ ]:
submission.to_csv('submission.csv', index=False)